In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [ ]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str


In [ ]:
def create_outline(state: BlogState) -> BlogState:
    prompt = f"Create a detailed outline for a blog post titled '{state['title']}'."
    outline = model.generate_text(prompt).content
    state['outline'] = outline
    return state

In [ ]:
def create_blog(state: BlogState) -> BlogState:
    prompt = f"Write a blog post based on the following outline:\n{state['outline']}"
    content = model.generate_text(prompt)
    state['content'] = content
    return state

In [ ]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [ ]:
initial_state = {'title': "The Future of AI", 'outline': "", 'content': ""}
final_state = workflow.invoke(initial_state)
print(final_state['content'])